# `mat_resolve`: Resolving a Set of Pairwise Image Registrations

## What problem are we solving?

Suppose you have **N brain images** and you have run pairwise registration between every pair — that is, for every combination of image $A$ and image $B$ you have a transformation matrix $\hat{T}_{A \to B}$ that aligns image $A$ to image $B$.

In a perfect world those matrices would be perfectly consistent with one another.  For example, the "going through image C" path

$$\hat{T}_{A \to C} \stackrel{?}{=} \hat{T}_{B \to C} \circ \hat{T}_{A \to B}$$

should give the same result as the direct $\hat{T}_{A \to C}$ registration.  In practice they never agree perfectly because every pairwise registration has a small error.

The goal of `mat_resolve` is to find a **single transform per image** — call it $\bar{T}_k$ for image $k$ — that maps the common *resolved* space into image $k$'s own space, chosen to be most consistent (in a least-squares sense) with all the observed pairwise matrices.

---
## Rigid-body transforms (6 degrees of freedom)

Every transform is a **rigid-body** motion: rotation + translation, no stretching or shearing.  In homogeneous coordinates it looks like a 4×4 matrix:

$$T = \begin{bmatrix} R & \mathbf{t} \\ \mathbf{0}^\top & 1 \end{bmatrix}$$

where $R$ is a 3×3 rotation matrix and $\mathbf{t}$ is a 3-vector of translations.

Because rigid bodies have exactly **6 degrees of freedom** (3 rotation, 3 translation), we can store each transform as a compact 6-vector instead of a full 4×4 matrix.  This is the **CK6 parameterization**:

$$\text{ck6} = [e_1,\; e_2,\; e_3,\; t_x,\; t_y,\; t_z]$$

where $(e_1, e_2, e_3)$ is the **vector part of a unit quaternion** that encodes the rotation, and $(t_x, t_y, t_z)$ is the translation.  The scalar part of the quaternion is recovered as:

$$e_0 = \sqrt{1 - e_1^2 - e_2^2 - e_3^2}$$

A quaternion $q = (e_0, e_1, e_2, e_3)$ with $e_0 \geq 0$ is a compact, non-singular way to represent a rotation for angles up to 180°.  The optimizer works directly with the 6-vector, which avoids the redundancy of storing all 12 entries of $(R, \mathbf{t})$.

---
## How the predicted pairwise transform is computed

Let $\bar{T}_k$ be the resolved transform for image $k$ (what we are solving for).  The internal convention is that $\bar{T}_k$ maps the common resolved space into image $k$'s own space:

$$\mathbf{x}_k = \bar{T}_k \cdot \mathbf{x}_{\text{resolved}}$$

One image is chosen as the reference (`resolve()` reorders the image list so it's always index 0); it is fixed at $\bar{T}_0 = I$, which pins the resolved space to be exactly the reference image's own coordinate space and removes the global ambiguity.

For an ordered pair of images, call the input image **moving** and the target image **fixed** — matching FSL's own `flirt -in <moving> -ref <fixed>` terminology.  Given the resolved transforms, the **predicted** alignment for that pair is:

$$T_{\text{moving} \to \text{fixed}}^{\text{pred}} = \bar{T}_{\text{moving}} \cdot \bar{T}_{\text{fixed}}^{-1}$$

Read right-to-left as the sequence of operations applied to a coordinate $\mathbf{x}$ in the *fixed* image's space:

1. $\bar{T}_{\text{fixed}}^{-1}$ maps $\mathbf{x}$ from the fixed image's space into the resolved space.
2. $\bar{T}_{\text{moving}}$ maps the result from the resolved space into the moving image's space.

so $T^{\text{pred}}_{\text{moving}\to\text{fixed}}$ satisfies $\mathbf{x}_{\text{moving}} = T^{\text{pred}} \cdot \mathbf{x}_{\text{fixed}}$.

### Where the observed data comes from — and why it has to be inverted

The observed data for a pair is whatever `flirt -in moving -ref fixed -omat pair.mat` writes to disk.  FSL's own convention for that file is the *opposite* direction from what we just derived:

$$\mathbf{x}_{\text{fixed}} = M_{\text{fsl}} \cdot \mathbf{x}_{\text{moving}}$$

so before comparing to the prediction, the raw file has to be inverted:

$$\hat{T}_{\text{moving}\to\text{fixed}} = M_{\text{fsl}}^{-1}$$

and it's $\hat{T}$ — not $M_{\text{fsl}}$ directly — that gets compared against $T^{\text{pred}}_{\text{moving}\to\text{fixed}}$.  This inversion happens at exactly three places in the code: reading an observed matrix (`resolve.py`'s `_build_residuals_and_jacobian`), seeding the initial guess from one (`_initialise_params`), and writing the final result back out as one (`transforms_io.py`'s `write_output`, which inverts the internal $\bar{T}_k$ back into FSL's convention so the output file is directly usable with `flirt -applyxfm`).

We want $T^{\text{pred}}_{\text{moving}\to\text{fixed}}$ to match $\hat{T}_{\text{moving}\to\text{fixed}}$ as closely as possible, for every ordered pair.

---
## What is being minimized

For each ordered (moving, fixed) pair with moving ≠ fixed we define a **6-dimensional residual vector** that measures how much $T_{\text{moving}\to\text{fixed}}^{\text{pred}}$ differs from $\hat{T}_{\text{moving}\to\text{fixed}}$.

The residual is split into a **rotation part** (3 numbers) and a **translation part** (3 numbers):

$$\mathbf{r}_{\text{moving},\text{fixed}} = \begin{bmatrix} \mathbf{r}^{\text{rot}}_{\text{moving},\text{fixed}} \\ \mathbf{r}^{\text{trans}}_{\text{moving},\text{fixed}} \end{bmatrix} \in \mathbb{R}^6$$

### Translation residual

The translation residual is simply the difference in translation vectors:

$$\mathbf{r}^{\text{trans}}_{\text{moving},\text{fixed}} = \mathbf{t}_{\text{moving}\to\text{fixed}}^{\text{pred}} - \hat{\mathbf{t}}_{\text{moving}\to\text{fixed}}$$

where $\mathbf{t}_{\text{moving}\to\text{fixed}}^{\text{pred}}$ is the translation column of $T_{\text{moving}\to\text{fixed}}^{\text{pred}}$, measured in millimetres.

### Rotation residual

Rotations cannot simply be subtracted.  Instead we measure their difference in quaternion space.  If the predicted rotation has quaternion $q^{\text{pred}}$ and the observed has $\hat{q}$, the rotation error is the quaternion that rotates one into the other:

$$q^{\text{err}} = \hat{q}^{-1} \cdot q^{\text{pred}}$$

When the prediction is perfect, $q^{\text{pred}} = \hat{q}$, so $q^{\text{err}} = (1, 0, 0, 0)$ (the identity rotation).  The rotation residual is the **vector part** of $q^{\text{err}}$, which is zero at the solution:

$$\mathbf{r}^{\text{rot}}_{\text{moving},\text{fixed}} = \text{vector part of } \left(\hat{q}_{\text{moving}\to\text{fixed}}^{-1} \cdot q_{\text{moving}\to\text{fixed}}^{\text{pred}}\right) \in \mathbb{R}^3$$

The vector part of a unit quaternion is approximately $\sin(\theta/2)\,\hat{n}$ where $\theta$ is the rotation angle and $\hat{n}$ is the axis, so for small errors this is proportional to the rotation angle.

### Scaling rotation to match translation units

The rotation residual is dimensionless (radians-like) while the translation residual is in millimetres (following FSL transformation matrix format).  To make them comparable we multiply the rotation residual by a **brain radius** $\rho$ (default 50 mm).  This converts the rotation error into an approximate displacement error at the surface of the brain:

$$\tilde{\mathbf{r}}^{\text{rot}}_{\text{moving},\text{fixed}} = \rho \cdot \mathbf{r}^{\text{rot}}_{\text{moving},\text{fixed}}$$

After this scaling, every element of every residual vector is in units of millimetres.

### The objective function

We stack all residuals into a single long vector $\mathbf{r}$ containing $6 \cdot N(N-1)$ elements (6 per ordered pair, $N(N-1)$ ordered pairs), and minimize its squared norm:

$$\min_{\bar{T}_1, \ldots, \bar{T}_{N-1}} \; \|\mathbf{r}\|^2 = \sum_{\text{moving} \neq \text{fixed}} \left( \rho^2 \|\mathbf{r}^{\text{rot}}_{\text{moving},\text{fixed}}\|^2 + \|\mathbf{r}^{\text{trans}}_{\text{moving},\text{fixed}}\|^2 \right)$$

Note that $\bar{T}_0 = I$ is **not** a free parameter — only $\bar{T}_1, \ldots, \bar{T}_{N-1}$ are optimized, giving $(N-1) \times 6$ free parameters in total.

---
## Worked example (N = 3)

### Setup

Three images: **img0** (the reference), **img1**, **img2**.

The true resolved transforms are:

| Image | Rotation | Translation |
|-------|----------|-------------|
| img0  | none (reference) | $[0, 0, 0]$ mm |
| img1  | 30° around the z-axis | $[0, 0, 0]$ mm |
| img2  | none | $[10, 0, 0]$ mm |

Written as 4×4 matrices (recall $\bar{T}_k$ maps the resolved space into image $k$'s coordinates: $\mathbf{x}_k = \bar T_k \cdot \mathbf{x}_{\text{resolved}}$):

$$\bar{T}_0 = I, \qquad
\bar{T}_1 = \begin{bmatrix} \tfrac{\sqrt{3}}{2} & -\tfrac{1}{2} & 0 & 0 \\ \tfrac{1}{2} & \tfrac{\sqrt{3}}{2} & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}, \qquad
\bar{T}_2 = \begin{bmatrix} 1 & 0 & 0 & 10 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

Because img0 is the reference, $\bar{T}_0 = I$ is fixed and never optimized.  The **free parameters** are $\bar{T}_1$ and $\bar{T}_2$ — 6 numbers each, 12 in total.

---

### The six raw FSL files this would produce

If you actually ran `flirt` on these three images, the six `.mat` files on disk would satisfy FSL's own convention, $\mathbf{x}_{\text{fixed}} = M_{\text{fsl}} \cdot \mathbf{x}_{\text{moving}}$ — the *opposite* direction from $\bar{T}_k$ above.  For a pair (moving, fixed), that raw matrix works out to $M_{\text{fsl}} = \bar{T}_{\text{fixed}} \cdot \bar{T}_{\text{moving}}^{-1}$.  For example, `img1_to_img0.mat` (moving=img1, fixed=img0) is $\bar{T}_0 \cdot \bar{T}_1^{-1} = \bar{T}_1^{-1}$, a $-30°$ rotation — the *inverse* of the 30° rotation in the table above, because the raw file maps img1's coordinates *back* into the reference frame, not the other way around.

The code cell below computes and prints all six raw files, verifies that the built-in loop-closure diagnostic (`symmetry.compute_symmetry_error`) reports zero error on them (as it must for perfectly consistent data), shows the inversion step for one pair explicitly (confirming the residual is exactly zero at the true solution), demonstrates what a small error in a raw file does to that residual, and finally runs the real `resolve()` end-to-end on this dataset to confirm it recovers the ground truth.

---

### What happens with noise?

Suppose the *raw* `img1_to_img2.mat` file has a small registration error: instead of the true translation column $[10, 0, 0]$, `flirt` reports $[9.5, -0.2, 0]$.  After inverting that file to get $\hat{T}_{\text{img1}\to\text{img2}}$ (as the optimizer does internally), this produces a small nonzero translation residual — computed explicitly below.  This residual is folded into the global least-squares sum $\|\mathbf{r}\|^2$ along with residuals from all other pairs.  The optimizer then finds the $\bar{T}_1, \bar{T}_2$ that minimize the total, spreading the error as evenly as possible across all constraints.

In [1]:
import numpy as np
from mat_resolve.rigid_math import rigid_inv, matrix_to_ck6, compose_rigid_with_jacobians
from mat_resolve.symmetry import compute_symmetry_error
from mat_resolve.resolve import resolve

deg = np.pi / 180
c, s = np.cos(30 * deg), np.sin(30 * deg)

# --- Ground truth Tbar[k]: resolved space -> image k's own space --------
Tbar = {
    "img0": np.eye(4),
    "img1": np.array([[c, -s, 0, 0], [s, c, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]),
    "img2": np.array([[1, 0, 0, 10], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]),
}
names = list(Tbar)

# --- The six raw FSL matrices: M_fsl = Tbar[fixed] @ inv(Tbar[moving]) ---
transforms = {}
for moving in names:
    for fixed in names:
        if moving == fixed:
            continue
        transforms[(moving, fixed)] = Tbar[fixed] @ rigid_inv(Tbar[moving])

print("Six raw FSL matrices ('{moving}_to_{fixed}.mat'), satisfying x_fixed = M @ x_moving:\n")
for (moving, fixed), M in transforms.items():
    print(f"{moving}_to_{fixed}.mat:\n{np.round(M, 4)}\n")

# --- Loop-closure check: the real pre-optimisation diagnostic -----------
print("--- Loop-closure check (uses the raw files directly, no inversion) ---")
compute_symmetry_error(transforms, names)

# --- The inversion step, and the residual it makes exactly zero ---------
print("\n--- Residual at the true solution for pair (moving=img1, fixed=img2) ---")
M_fsl = transforms[("img1", "img2")]
T_hat = rigid_inv(M_fsl)
print("raw img1_to_img2.mat:\n", np.round(M_fsl, 4))
print("inverted -> T_hat (used internally):\n", np.round(T_hat, 4))

ck6_moving, ck6_fixed = matrix_to_ck6(Tbar["img1"]), matrix_to_ck6(Tbar["img2"])
_, residual, _, _ = compose_rigid_with_jacobians(ck6_moving, ck6_fixed, matrix_to_ck6(T_hat))
print("residual at truth:", np.round(residual, 10))

# --- Same pair, with a small error injected into the raw file -----------
print("\n--- Same pair, with a small error injected into the raw file ---")
M_fsl_noisy = M_fsl.copy()
M_fsl_noisy[:3, 3] = [9.5, -0.2, 0]  # true value was [10, 0, 0]
ck6_hat_noisy = matrix_to_ck6(rigid_inv(M_fsl_noisy))
_, residual_noisy, _, _ = compose_rigid_with_jacobians(ck6_moving, ck6_fixed, ck6_hat_noisy)
print("residual with noisy raw file:", np.round(residual_noisy, 4))

# --- Full resolve() round-trip recovers the ground truth exactly --------
print("\n--- Full resolve() round-trip recovers the ground truth exactly ---")
result = resolve(transforms, names)
for name in names:
    err = np.max(np.abs(result.resolved[name] - rigid_inv(Tbar[name])))
    print(f"{name}: max abs error vs ground truth = {err:.2e}")
print("final_rms:", result.final_rms)

Six raw FSL matrices ('{moving}_to_{fixed}.mat'), satisfying x_fixed = M @ x_moving:

img0_to_img1.mat:
[[ 0.866 -0.5    0.     0.   ]
 [ 0.5    0.866  0.     0.   ]
 [ 0.     0.     1.     0.   ]
 [ 0.     0.     0.     1.   ]]

img0_to_img2.mat:
[[ 1.  0.  0. 10.]
 [ 0.  1.  0.  0.]
 [ 0.  0.  1.  0.]
 [ 0.  0.  0.  1.]]

img1_to_img0.mat:
[[ 0.866  0.5    0.     0.   ]
 [-0.5    0.866  0.     0.   ]
 [ 0.     0.     1.     0.   ]
 [ 0.     0.     0.     1.   ]]

img1_to_img2.mat:
[[ 0.866  0.5    0.    10.   ]
 [-0.5    0.866  0.     0.   ]
 [ 0.     0.     1.     0.   ]
 [ 0.     0.     0.     1.   ]]

img2_to_img0.mat:
[[  1.   0.   0. -10.]
 [  0.   1.   0.   0.]
 [  0.   0.   1.   0.]
 [  0.   0.   0.   1.]]

img2_to_img1.mat:
[[ 0.866  -0.5     0.     -8.6603]
 [ 0.5     0.866   0.     -5.    ]
 [ 0.      0.      1.      0.    ]
 [ 0.      0.      0.      1.    ]]

--- Loop-closure check (uses the raw files directly, no inversion) ---

Symmetry error over 6 ordered pairs:
  Rot

---
## Initialization

The optimizer needs a starting point.  We initialize each $\bar{T}_k$ from the directly observed registration of image $k$ to the reference (image 0), inverting the raw FSL file the same way as during optimization:

$$\bar{T}_k^{(0)} = \hat{T}_{k \to 0} = M_{\text{fsl},\,k\to0}^{-1}$$

This is already a reasonable answer — the optimization then adjusts these transforms to make them collectively self-consistent across **all** pairs, not just the direct-to-reference registrations.

---
## The Jacobian (why we provide it analytically)

The optimizer (`scipy.optimize.least_squares` with the Levenberg–Marquardt method) needs to know not just the residuals but also how they change as the parameters change.  This is the **Jacobian matrix** $J$, assembled by `resolve.py`'s `_build_residuals_and_jacobian`:

$$J_{(\text{moving},\text{fixed}),\,k} = \frac{\partial \mathbf{r}_{\text{moving},\text{fixed}}}{\partial \bar{T}_k}$$

For an $N=4$ image problem this matrix has shape $6 \cdot 4 \cdot 3 = 72$ rows and $3 \cdot 6 = 18$ columns.

Most entries are zero because the residual for a (moving, fixed) pair only depends on the parameters of those two images.  The non-zero blocks come from two chain-rule steps:

1. How does the predicted quaternion $q^{\text{pred}}$ change when the rotation parameters of the moving (or fixed) image change?
2. How does the rotation residual $\mathbf{r}^{\text{rot}}$ change when $q^{\text{pred}}$ changes?

Both steps have closed-form formulas (ported from the original Fortran library `CKsub.f` / `getjxy.f`), so we supply the Jacobian analytically rather than letting the optimizer estimate it numerically.  This makes convergence faster and more reliable.

---
## Deriving $\partial R / \partial e_k$ (`quaternion_to_rotation_with_jacobian`)

The rotation matrix built from a unit quaternion $q = (e_0, e_1, e_2, e_3)$ has a well-known closed form.  The **Fortran convention** used by `CKsub.f` (call it $R_f$, related to scipy's matrix by $R_f = R^\top$) is:

$$R_f = \begin{bmatrix}
1-2(e_2^2+e_3^2) & 2(e_1e_2+e_0e_3) & 2(e_1e_3-e_0e_2) \\
2(e_1e_2-e_0e_3) & 1-2(e_1^2+e_3^2) & 2(e_2e_3+e_0e_1) \\
2(e_1e_3+e_0e_2) & 2(e_2e_3-e_0e_1) & 1-2(e_1^2+e_2^2)
\end{bmatrix}$$

Every entry is an explicit polynomial in $e_0, e_1, e_2, e_3$.  Only $e_1, e_2, e_3$ are free parameters, though — $e_0$ is **not** independent, it's defined by the constraint $e_0 = \sqrt{1-e_1^2-e_2^2-e_3^2}$.  So every place $e_0$ appears in an $R_f$ entry, it's really shorthand for that whole square-root expression in $e_1,e_2,e_3$.  This means differentiating with respect to $e_k$ has to account for $e_k$ wherever it appears — explicitly, *and* hidden inside every $e_0$ — via the ordinary chain rule:

$$\frac{\partial e_0}{\partial e_k} = -\frac{e_k}{e_0}$$

**This is the crucial step that's easy to skip.** If you treat $e_0$ as if it were just another constant while differentiating, you'll drop a term.  Concretely, take $R_f[0,1] = 2(e_1e_2+e_0e_3)$, written out fully as a function of the three free variables:

$$R_f[0,1](e_1,e_2,e_3) = 2e_1e_2 + 2e_3\underbrace{\sqrt{1-e_1^2-e_2^2-e_3^2}}_{e_0}$$

Differentiating with respect to $e_1$ means differentiating *both* terms — the explicit $e_1$ in $2e_1e_2$, and the hidden $e_1$ buried inside $e_0$ in the second term. The first gives the "obvious" piece $2e_2$; the second requires plugging in $\partial e_0/\partial e_1 = -e_1/e_0$ (derived above) via the chain rule:

$$\frac{\partial R_f[0,1]}{\partial e_1} = \underbrace{2e_2}_{\text{explicit }e_1} + \underbrace{2e_3\cdot\frac{\partial e_0}{\partial e_1}}_{\text{hidden }e_1\text{ inside }e_0} = 2e_2 - \frac{2e_1e_3}{e_0}$$

Forgetting the second term (i.e. writing $\partial R_f[0,1]/\partial e_1 = 2e_2$) is the natural mistake to make here, since $e_0$ *looks* like a separate symbol — but it's a stand-in for a function of $e_1$, so its own derivative can't be dropped.

The code doesn't leave the result in this form — it rewrites it in terms of an *already-computed* $R_f$ entry divided by $e_0$, which is both more compact and cheaper to evaluate.  Notice $R_f[0,2] = 2(e_1e_3 - e_0e_2)$, so:

$$-\frac{R_f[0,2]}{e_0} = -\frac{2e_1e_3}{e_0} + 2e_2 = \frac{\partial R_f[0,1]}{\partial e_1}$$

which is exactly the line `dR_f[0, 1, 0] = -Rf[0, 2] / e0`. Every other entry in the `dR_f[...,0]`, `dR_f[...,1]`, `dR_f[...,2]` blocks is derived the same way: differentiate the explicit polynomial, apply $\partial e_0/\partial e_k = -e_k/e_0$ wherever $e_0$ appears, then use the unit-norm identity $e_0^2+e_1^2+e_2^2+e_3^2=1$ to fold sums of squares back into other $R_f$ entries (this is what happens for the diagonal terms, e.g. `dR_f[1,2,0] = (Rf[1,1]+Rf[2,2])/e0`, where $R_f[1,1]+R_f[2,2] = 2e_0^2-2e_1^2$ after substituting the identity). Terms with no $e_0$-dependence at all differentiate directly, e.g. `dR_f[1,1,0] = -4*e1` from $-2e_1^2$ in $R_f[1,1]$.

Finally, since `quaternion_to_rotation` returns scipy's convention $R = R_f^\top$, the derivative array is transposed on its first two axes to match:

$$\text{dR}[i,j,k] = \text{dR\_f}[j,i,k]$$

which is the `dR = dR_f.transpose(1, 0, 2)` line at the end of the function.</cell id="b985b262">


---
## Deriving the Hamilton product Jacobian (`quaternion_multiply_with_jacobians`)

The Hamilton product $g = e \cdot f$ of $e=(e_0,e_1,e_2,e_3)$ and $f=(f_0,f_1,f_2,f_3)$ can be written as a matrix-vector product two equivalent ways, using the standard **left-** and **right-multiplication matrices** of a quaternion:

$$g_{\text{full}} = L_e \cdot f_{\text{full}} = R_f \cdot e_{\text{full}}$$

$$L_e = \begin{bmatrix} e_0 & -e_1 & -e_2 & -e_3 \\ e_1 & e_0 & -e_3 & e_2 \\ e_2 & e_3 & e_0 & -e_1 \\ e_3 & -e_2 & e_1 & e_0 \end{bmatrix}, \qquad
R_f = \begin{bmatrix} f_0 & -f_1 & -f_2 & -f_3 \\ f_1 & f_0 & f_3 & -f_2 \\ f_2 & -f_3 & f_0 & f_1 \\ f_3 & f_2 & -f_1 & f_0 \end{bmatrix}$$

We need $\partial g_{\text{full}} / \partial e_j$ and $\partial g_{\text{full}} / \partial f_j$ (for $j=1,2,3$), with $e_0$ (resp. $f_0$) an implicit function of $e_1,e_2,e_3$ (resp. $f_1,f_2,f_3$) via $\partial e_0/\partial e_j = -e_j/e_0$, same as in the rotation-matrix derivation above.

**The key simplification**: since $g_{\text{full}} = L_e \cdot f_{\text{full}}$ is *linear* in $e$ with $f_{\text{full}}$ held fixed, the product rule needs no term for differentiating $L_e$ itself — differentiating $g_{\text{full}} = R_f \cdot e_{\text{full}}$ with respect to $e_j$ (holding $f$, and hence $R_f$, fixed) only requires differentiating $e_{\text{full}}$:

$$\frac{\partial g_{\text{full}}}{\partial e_j} = R_f \cdot \frac{\partial e_{\text{full}}}{\partial e_j}$$

and symmetrically, differentiating $g_{\text{full}} = L_e \cdot f_{\text{full}}$ with respect to $f_j$ only requires differentiating $f_{\text{full}}$:

$$\frac{\partial g_{\text{full}}}{\partial f_j} = L_e \cdot \frac{\partial f_{\text{full}}}{\partial f_j}$$

Each of $\partial e_{\text{full}}/\partial e_j$ and $\partial f_{\text{full}}/\partial f_j$ is a simple 4-vector, entirely determined by the implicit-function relationship for the scalar part and the trivial derivative for the vector part:

$$\frac{\partial e_{\text{full}}}{\partial e_j} = \left(-\frac{e_j}{e_0},\; \delta_{1j},\; \delta_{2j},\; \delta_{3j}\right)$$

(and the same form for $f$).  This is exactly `dg_de[:, k] = R_f @ de_dek` and `dg_df[:, k] = L_e @ df_dfk` in the code, where `de_dek`/`df_dfk` are built from this 4-vector formula in a loop over `k in range(3)`.

Compare this to differentiating $g$ written out as nine explicit cross terms by hand (as the original Fortran `ckmuld` does, via a Levi-Civita-style lookup table of which basis index and sign appears in each off-diagonal entry) — packaging the product as two constant 4×4 matrices sidesteps that bookkeeping entirely: one matrix ($R_f$ or $L_e$) absorbs *all* of the cross terms at once, and only the implicit $e_0$/$f_0$ dependence needs to be differentiated by hand.  The two forms are verified to produce identical output.

---
## Combining the pieces: `compose_rigid_with_jacobians` (`getjxy`)

This function assembles the two building blocks above into $J_{\text{moving}}$ and $J_{\text{fixed}}$, the $6\times6$ Jacobians of the residual with respect to the moving and fixed images' CK6 parameters — for the pair whose predicted transform is $T^{\text{pred}} = \bar{T}_{\text{moving}} \cdot \bar{T}_{\text{fixed}}^{-1}$ (see the "How the predicted pairwise transform is computed" section above; `ck6_hat` is already the *inverted* raw FSL matrix by the time it reaches this function — the inversion itself happens one level up, in `resolve.py`).

### Rotation residual

The predicted rotation is $q_{\text{pred}} = q_{\text{moving}} \cdot q_{\text{fixed}}^{-1}$.  Because the vector part alone determines a unit quaternion's inverse-as-conjugate, `e_fixed_inv = -ck6_fixed[:3]` already *is* the vector part of $q_{\text{fixed}}^{-1}$ — negating the vector part leaves $e_0=\sqrt{1-\|e\|^2}$ unchanged, exactly matching quaternion conjugation.

The rotation residual is the vector part of $\hat{q}^{-1}\cdot q_{\text{pred}}$ — i.e. "left-multiply $q_{\text{pred}}$ by $\hat{q}^{-1}$".  Left quaternion multiplication is a *linear* map, so it can be written as a $4\times4$ matrix acting on the 4-vector $(g_0,g_1,g_2,g_3)$ — exactly the way a cross product $a \times (\cdot)$ can be written as a skew-symmetric matrix.  That matrix is `E4hat`, built once from the fixed observed quaternion $\hat q$; the code only ever forms its bottom three rows (`E4hat_vec`) since only the vector part of the product is needed:

$$\mathbf{r}^{\text{rot}} = \texttt{E4hat\_vec} \cdot q_{\text{pred}}^{\text{full}}$$

Because `E4hat_vec` does **not** depend on $e_{\text{moving}}$ or $e_{\text{fixed}}$ (it's built entirely from the fixed, observed $\hat q$), differentiating is trivial — no product rule needed, just push the (already-known) Jacobian of $q_{\text{pred}}^{\text{full}}$ through the constant matrix:

$$\frac{\partial \mathbf{r}^{\text{rot}}}{\partial e_{\text{moving}}} = \texttt{E4hat\_vec} \cdot \frac{\partial q_{\text{pred}}^{\text{full}}}{\partial e_{\text{moving}}} = \texttt{E4hat\_vec} \cdot \texttt{dqpred\_dmoving}$$

which is `J_moving[:3, :3]`. For `J_fixed` there's one extra chain-rule factor from $e_{\text{fixed\_inv}} = -e_{\text{fixed}}$, contributing a $-1$:

$$\frac{\partial \mathbf{r}^{\text{rot}}}{\partial e_{\text{fixed}}} = -\bigl(\texttt{E4hat\_vec} \cdot \texttt{dqpred\_dfixed\_inv}\bigr)$$

which is `J_fixed[:3, :3]`.

### Translation residual

The predicted translation is $t_{\text{pred}} = t_{\text{moving}} - R_{\text{pred}}\,t_{\text{fixed}}$ with $R_{\text{pred}}=R_{\text{moving}}R_{\text{fixed\_inv}}$. This is now an ordinary product-rule differentiation of a matrix-vector product:

- **w.r.t. $e_{\text{moving}}$**: only $R_{\text{moving}}$ (inside $R_{\text{pred}}$) depends on $e_{\text{moving}}$, so $\partial R_{\text{pred}}/\partial e_{\text{moving},k} = (\partial R_{\text{moving}}/\partial e_{\text{moving},k})\,R_{\text{fixed\_inv}}$ (computed as `dR_pred_dmoving` via `einsum`), giving $\partial t_{\text{pred}}/\partial e_{\text{moving},k} = -(\partial R_{\text{pred}}/\partial e_{\text{moving},k})\,t_{\text{fixed}}$ — the `J_moving[3:, k]` loop.
- **w.r.t. $e_{\text{fixed}}$**: $R_{\text{fixed\_inv}}$ depends on $e_{\text{fixed\_inv}}=-e_{\text{fixed}}$, contributing a $-1$ from the chain rule, but that cancels against the $-1$ already in front of $R_{\text{pred}}\,t_{\text{fixed}}$ in $t_{\text{pred}}$'s definition — the two minus signs cancel, leaving `J_fixed[3:, k] = dR_pred_dfixed_inv[:, :, k] @ t_fixed` with **no** leading minus sign (see the code comment spelling out this cancellation explicitly).
- **w.r.t. $t_{\text{moving}}$**: $t_{\text{moving}}$ appears directly and linearly, so $\partial t_{\text{pred}}/\partial t_{\text{moving}} = I$ — `J_moving[3:, 3:] = np.eye(3)`.
- **w.r.t. $t_{\text{fixed}}$**: $t_{\text{fixed}}$ appears directly through $-R_{\text{pred}}\,t_{\text{fixed}}$, so $\partial t_{\text{pred}}/\partial t_{\text{fixed}} = -R_{\text{pred}}$ — `J_fixed[3:, 3:] = -R_predicted`.

Every one of these eight blocks is checked against numerical finite differences below.

In [2]:
import numpy as np
from mat_resolve.rigid_math import (
    quaternion_to_rotation,
    quaternion_to_rotation_with_jacobian,
    compose_rigid_with_jacobians,
)

rng = np.random.default_rng(0)
h = 1e-6


def random_e():
    return rng.uniform(-0.3, 0.3, size=3)


def random_ck6():
    return np.concatenate([random_e(), rng.uniform(-10, 10, size=3)])


# --- Finite-difference check of dR/de (quaternion_to_rotation_with_jacobian) --
e_vec = random_e()
_, dR_analytic = quaternion_to_rotation_with_jacobian(e_vec)

dR_fd = np.zeros((3, 3, 3))
for k in range(3):
    e_plus, e_minus = e_vec.copy(), e_vec.copy()
    e_plus[k] += h
    e_minus[k] -= h
    dR_fd[:, :, k] = (
        quaternion_to_rotation(e_plus) - quaternion_to_rotation(e_minus)
    ) / (2 * h)

print("dR/de       max abs error (analytic vs finite-difference):",
      np.max(np.abs(dR_fd - dR_analytic)))

# --- Finite-difference check of J_moving, J_fixed (compose_rigid_with_jacobians) ----
ck6_moving, ck6_fixed, ck6_hat = random_ck6(), random_ck6(), random_ck6()
_, _, J_moving, J_fixed = compose_rigid_with_jacobians(ck6_moving, ck6_fixed, ck6_hat)


def residual(a, b):
    _, r, _, _ = compose_rigid_with_jacobians(a, b, ck6_hat)
    return r


J_moving_fd = np.zeros((6, 6))
J_fixed_fd = np.zeros((6, 6))
for k in range(6):
    d = np.zeros(6)
    d[k] = h
    J_moving_fd[:, k] = (residual(ck6_moving + d, ck6_fixed) - residual(ck6_moving - d, ck6_fixed)) / (2 * h)
    J_fixed_fd[:, k] = (residual(ck6_moving, ck6_fixed + d) - residual(ck6_moving, ck6_fixed - d)) / (2 * h)

print("J_moving    max abs error (analytic vs finite-difference):",
      np.max(np.abs(J_moving_fd - J_moving)))
print("J_fixed     max abs error (analytic vs finite-difference):",
      np.max(np.abs(J_fixed_fd - J_fixed)))

dR/de       max abs error (analytic vs finite-difference): 1.242443925519865e-10
J_moving    max abs error (analytic vs finite-difference): 8.324226863365425e-10
J_fixed     max abs error (analytic vs finite-difference): 1.3492034156570298e-09


---
## What the optimizer returns

`resolve()` returns a `ResolveResult` holding one 4×4 matrix per image, already inverted back into FSL's own convention (`result.resolved[name]`, directly usable with `flirt -applyxfm`), plus `initial_rms`/`final_rms`/`per_frame_rms`.  From there, `cli.py` (via `transforms_io.py`):

1. Writes one FSL `.mat` file per image (`{image}_to_{output}.mat`).
2. Prints a summary table of rotation (degrees) and translation (mm) for each image, plus its per-frame RMS residual.
3. Optionally resamples each image into the resolved space using `flirt -applyxfm`.

Before any of that, `cli.py` also prints a **symmetry error report** (`symmetry.compute_symmetry_error`) that measures the loop-closure consistency of the *input* pairwise registrations, straight off disk, before optimization — useful for spotting a badly failed registration that should be excluded.  This is a diagnostic only; it plays no role in the optimization itself.

### Quick sanity check: what does "resolved" mean?

If all the pairwise registrations are perfectly consistent (no noise), every residual will be driven to exactly zero and the recovered $\bar{T}_k$ will match the ground-truth transforms to floating-point precision.  With real (noisy) registrations the optimizer finds the least-squares best compromise, essentially averaging out the pairwise errors.

---
*For implementation details see `mat_resolve/rigid_math.py` (quaternion math and Jacobians), `mat_resolve/resolve.py` (residual/Jacobian assembly and the `resolve()` API), `mat_resolve/transforms_io.py` (file discovery, I/O, and `flirt` interaction), `mat_resolve/cli.py` (thin argument-parsing wrapper around the above), and `mat_resolve/symmetry.py` (loop-closure error report).*